# 特許調査パイプライン（ステップ1〜3）
1. 指定フォルダ内の J-PlatPat 検索結果一覧（コピーしたテキスト、複数可）を統合し、重複を除いた文献番号表を作る
2. 文献番号から Google Patents（日本語ページ）の HTML を取得し、名称・要約・請求項・明細書を Excel に出力する
3. ローカル LLM で、各特許が「調べたいこと」に関連するかを評価して表にする

**使い方**: 上から順に実行。まず設定セルの値を変更し、手順2・3は `LIMIT` / `EVAL_LIMIT = 3` などで少数件を試してから全件にしてください。

**各セルの説明は、そのセルの上にあります。** プログラム本体のセル（セル3・8・11）は、先頭にそのプログラムの作業の流れが書いてあります。

注意: Google Patents の自動取得は利用規約上の制限を受け得ます。`DELAY` を空けて、必要な件数だけ実行してください。

### セル1: ライブラリの準備
- **作業**: 必要な Python ライブラリ(表計算・Excel 出力・通信・HTML 解析)を入れる。入っていれば何もしない
- **入力**: なし
- **出力**: なし(ライブラリが使える状態になる)

In [ ]:
%pip install -q pandas openpyxl requests beautifulsoup4 lxml

### セル2: 設定（調べたいことはここに書く）
- **作業**: フォルダ・出力先・件数などの設定と、手順3の「調べたいこと」、使う LLM を決める。各設定の上にその意味を書いてある
- **入力**: あなたが書き換える値
- **出力**: なし(以降のセルが、ここの値を使う)

In [ ]:
from pathlib import Path
import json

# ================= 手順1・2 の設定 =================

# J-PlatPat の結果一覧をコピーした txt を入れたフォルダ。複数ファイルを置くと、まとめて読み込んで重複を除く。
INPUT_DIR = "../data"
# フォルダ内で読み込むファイルの名前の形。
INPUT_PATTERN = "*.txt"
# 手順1の出力: 重複を除いた文献番号の一覧(Excel)。
NUMBERS_XLSX = "results/patent_numbers.xlsx"
# 手順2の出力: 名称・要約・請求項・明細書の表(Excel)。
RESULT_XLSX = "results/patents_text.xlsx"
# 取得した生の HTML の保存先。再実行時は、ここにあるものは取得し直さない。
HTML_DIR = "results/html"
# 全文 JSON の保存先。Excel は約3万字で切れるため、LLM 評価にはこちらを使う。
TEXT_DIR = "results/text"
# 手順2で処理する件数。まず 3 で試し、問題なければ None(全件)にする。
LIMIT = 3
# Google Patents へのアクセス間隔(秒)。短くしすぎない。
DELAY = 3.0

# ================= 手順3 の設定: 調べたいこと（ここを書き換える。プロンプトにそのまま組み込まれる）=================

# テーマの名前。結果の表やプロンプトに表示される。
TOPIC_NAME = "TOTAL-RNA-seq"

# 何を指すテーマか。専門外の人にも通じる文章で、手法・対象・範囲を定義する。
TOPIC_DEFINITION = """
TOTAL-RNA-seq（全RNAシーケンシング）とは、poly(A)選択でmRNAだけを濃縮するのではなく、
rRNA除去（rRNA depletion / ribo-depletion）などによって、mRNAに加えてlncRNA・非ポリA RNA・
前駆体RNAなどを含む全RNAを対象にライブラリを作製し、次世代シーケンサーで読み取る手法をいう。
"""

# 同義語・日英表記・商品名。本文中の出現箇所を抜粋してLLMに渡す（網羅的なほど拾いやすい）。
TOPIC_KEYWORDS = [
    "total RNA-seq", "total RNA sequencing", "total RNA", "whole transcriptome sequencing",
    "rRNA depletion", "ribosomal RNA depletion", "ribo-depletion", "Ribo-Zero", "RiboMinus",
    "全RNA", "トータルRNA", "全トランスクリプトーム", "rRNA除去", "リボソームRNA除去",
]

# 関連とみなす条件（任意）。判断が分かれる境界を文章で決めておく。
TOPIC_INCLUDE = """
total RNA-seq、rRNA除去を伴うRNA-seq、非コードRNAや前駆体RNAも含めた網羅的RNA配列解析の
手法・キット・データ解析方法。それらを実施例で実際に使っている特許も含める。
"""

# 関連とみなさない条件（任意）。
TOPIC_EXCLUDE = """
poly(A)選択によるmRNA-seqのみを扱い、total RNAやrRNA除去に触れないもの。
RNAを単に治療薬の成分として扱うもの（核酸医薬、mRNAワクチンなど）で、RNA-seq解析を含まないもの。
"""

# ================= 手順3 の設定: LLM =================

# 使う LLM の方式。"ollama"(この PC) または "guidance"(サーバー: Hugging Face のモデル)。
BACKEND = "ollama"
# Ollama のモデル名。ollama list で表示される名前。
OLLAMA_MODEL = "qwen3:14b"
# Ollama が一度に読める長さ。大きいと遅くメモリも使う。MAX_CHARS に合わせる。
OLLAMA_NUM_CTX = 8192
# guidance 用の Hugging Face のモデル ID。サーバーでは使うモデルに変更する。
HF_MODEL_ID = "Qwen/Qwen3-14B"
# 1件あたり LLM に渡す本文の最大文字数（請求項と明細書で半分ずつ）。
MAX_CHARS = 6000
# 手順3で評価する件数。まず 3 で試し、問題なければ None(全件)にする。
EVAL_LIMIT = 3
# 手順3の出力: 関連性の評価結果(Excel)。文献番号は Google Patents へのリンクになる。
RESULT_EVAL_XLSX = "results/evaluation.xlsx"

# 出願企業の表示名を統一する対応表。{正規化した名前: 表示名}。
# 正規化した名前は、全角半角をそろえ、空白・中黒と「株式会社」「インコーポレイテッド」等の法人格を除いたもの。
# 例: "エルジー　エレクトロニクス　インコーポレイティド" -> "エルジーエレクトロニクス"。必要に応じて追加する。
COMPANY_ALIASES = {
    "三星電子": "Samsung Electronics",
    "三星エスディアイ": "Samsung SDI",
    "エルジーエレクトロニクス": "LG Electronics",
    "エルジーエナジーソリューション": "LG Energy Solution",
    "エルジーケム": "LG Chem",
    "エルジーディスプレイ": "LG Display",
    "シェイプコープ": "Shape Corp.",
}

# 出力先フォルダを作る(すでにあれば何もしない)。
Path("results").mkdir(exist_ok=True)

## 手順1: 文献番号の一覧を作る

### セル3: プログラム1: テキストの読み取り
- **作業**: J-PlatPat の結果一覧テキストを 1 件ずつに区切り、文献番号・出願番号・日付・名称・出願人・ステータス・FI に振り分ける。フォルダ内の複数ファイルをまとめ、文献番号で重複を除く。Google Patents の ID 候補も作る
- **入力**: なし(関数を定義するだけ)
- **出力**: なし(次のセルで使う関数 parse_text・build_list などができる)

In [ ]:
"""プログラム1: J-PlatPat「検索結果一覧(国内文献)」のコピーテキストを、重複のない表にする。

J-PlatPat の画面は JavaScript で描画されるため、保存した HTML には表が入らない。
そのため、結果一覧をコピーして貼り付けたテキスト(*.txt)を読む。

作業の流れ:
 1. parse_text      : テキストを 1 件ずつ(No. の連番を目印に)区切り、各項目に振り分ける
 2. google_patent_id: 文献番号から Google Patents の ID 候補(例: JP2017080742A)を作る
 3. build_list      : フォルダ内の全 txt を読み、文献番号で重複を除いた一覧(DataFrame)にまとめる
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

FI_RE = re.compile(r"^[A-HY]\d{2}[A-Z]\d+")
DATE_RE = re.compile(r"^\d{4}/\d{2}/\d{2}$")
DOC_RE = re.compile(
    r"^(?P<kind>特開|特表|再表|特許|実登|実公|実開|公開実用|登録実用|特公)"
    r"(?P<era>昭|平)?(?P<body>[\d\-/]+)$"
)
# 文献種別 -> Google Patents の種別コード（推定。B1/B2 や Y1/Y2 は判別できない）
KIND_CODE = {"特開": "A", "特表": "A", "再表": "A1", "特許": "B2",
             "実登": "U", "実開": "U", "公開実用": "U", "登録実用": "U",
             "実公": "Y2", "特公": "B2"}
STATUS_STARTS = ("審査", "特許", "（出願の）", "-", "登録", "権利", "実用", "拒絶")
HEADER_END = "FI"


@dataclass
class PatentRow:
    no: int
    doc_no: str = ""
    app_no: str = ""
    filing_date: str = ""
    publication_date: str = ""
    title: str = ""
    applicant: str = ""
    applicant_has_more: bool = False
    status: list[str] = field(default_factory=list)
    fi: list[str] = field(default_factory=list)
    fi_has_more: bool = False
    warnings: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["status"] = " / ".join(self.status)
        d["fi"] = " ; ".join(self.fi)
        d["warnings"] = " ; ".join(self.warnings)
        d["google_patent_id"] = google_patent_id(self.doc_no)
        d["google_patent_url"] = (
            f"https://patents.google.com/patent/{d['google_patent_id']}/ja"
            if d["google_patent_id"] else ""
        )
        return d


# 作業: 文献番号(特開・特表・特許・実登・再表 など)を Google Patents の ID 候補に変換する。
def google_patent_id(doc_no: str) -> str:
    """文献番号から Google Patents の ID 候補を作る（種別コードは推定）。

    例: 特開2026-139649 -> JP2026139649A / 特開昭50-051518 -> JPS50051518A
    """
    m = DOC_RE.match(doc_no)
    if not m:
        return ""
    # 再表(国際出願の再公表)は特殊な形式にする。
    # 例: 再表2015/072306 -> JPWO2015072306A1, 再表92/019759 -> JPWO1992019759A1
    if m["kind"] == "再表":
        y, n = m["body"].split("/")
        y = y if len(y) == 4 else ("19" if int(y) >= 50 else "20") + y
        return f"JPWO{y}{n}A1"
    era = {"昭": "S", "平": "H", None: ""}[m["era"]]
    digits = m["body"].replace("-", "")
    return f"JP{era}{digits}{KIND_CODE[m['kind']]}"


def _is_status_line(line: str) -> bool:
    return line.startswith(STATUS_STARTS)


# 作業: コピーテキスト全体を 1 件ごとに区切って PatentRow のリストにする。
def parse_text(text: str) -> list[PatentRow]:
    lines = [l.strip() for l in text.splitlines()]
    lines = [l for l in lines if l]
    # ヘッダ（"FI" まで）を飛ばす
    try:
        start = lines.index(HEADER_END) + 1
    except ValueError:
        start = 0
    body = lines[start:]

    # レコード境界: 連番 1,2,3... と一致する数字だけの行
    # （文献番号は数字だけにならないため、連番との一致で十分に区別できる）
    starts, expect = [], 1
    for i, l in enumerate(body):
        if l.isdigit() and int(l) == expect:
            starts.append(i)
            expect += 1
    rows = []
    for k, s in enumerate(starts):
        e = starts[k + 1] if k + 1 < len(starts) else len(body)
        chunk = body[s + 1:e]
        # 最後の 1 件だけ、末尾の著作権表示などのフッタ行を取り除く
        if k + 1 == len(starts):
            chunk = [l for l in chunk if not l.startswith(("Copyright", "(P"))]
        rows.append(_parse_record(k + 1, chunk))
    return rows


# 作業: 1 件分の行(文献番号、出願番号、日付、名称、出願人、ステータス、FI)を各項目に振り分ける。
def _parse_record(no: int, chunk: list[str]) -> PatentRow:
    r = PatentRow(no=no)
    if len(chunk) < 6:
        r.warnings.append("行数不足")
        return r
    r.doc_no, r.app_no, r.filing_date, r.publication_date, r.title, r.applicant = chunk[:6]
    rest = chunk[6:]
    if not DOC_RE.match(r.doc_no):
        r.warnings.append("文献番号の書式が未知")
    for label, v in (("出願日", r.filing_date), ("公知日", r.publication_date)):
        if not DATE_RE.match(v):
            r.warnings.append(f"{label}の書式が未知")
    # 出願人欄の「他」（FI の「他」と区別するため、FI より前にあるものだけ）
    i = 0
    if rest and rest[0] == "他":
        r.applicant_has_more = True
        i = 1
    for l in rest[i:]:
        if l == "他":
            r.fi_has_more = True
        elif FI_RE.match(l) or re.match(r"^[A-HY]\d{2}[A-Z]\d+", l):
            r.fi.append(l)
        elif _is_status_line(l) or not r.fi:
            r.status.append(l)
        else:
            r.warnings.append(f"分類不能な行: {l}")
    return r


# 作業: 1 つの txt ファイルを読んで parse_text に渡す。
def load_file(path: str | Path) -> list[PatentRow]:
    return parse_text(Path(path).read_text(encoding="utf-8"))


# ---- フォルダ内の複数ファイルを統合し、重複を除いた一覧を作る ------------------------------------

LIST_COLUMNS = {
    "no": "No.", "doc_no": "文献番号", "app_no": "出願番号", "filing_date": "出願日",
    "publication_date": "公知日", "title": "発明の名称", "applicant": "出願人/権利者",
    "applicant_has_more": "出願人(他あり)", "status": "ステータス", "fi": "FI",
    "fi_has_more": "FI(他あり)", "google_patent_id": "Google Patents ID(推定)",
    "google_patent_url": "Google Patents URL(推定)", "warnings": "警告",
}


def read_text_file(path: Path) -> str:
    """UTF-8(BOM可)で読み、失敗したら Shift_JIS(cp932)で読む。"""
    raw = Path(path).read_bytes()
    for enc in ("utf-8-sig", "cp932"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def build_list(folder: str | Path, pattern: str = "*.txt"):
    """folder 内の pattern に合う全ファイルを読み、文献番号で重複を除いた一覧を作る。

    戻り値: (一覧 DataFrame, 報告 dict)
    - 同じ文献番号が複数のファイルにある場合は、更新日時が新しいファイルの行を採用する
      （ステータスは時間とともに変わるため）。出典は「出典ファイル」列に全て残す。
    - 同じ出願番号で文献番号が異なる行(公開公報と登録公報など)は除かず、「同一出願番号の別文献」列で示す。
    """
    import pandas as pd

    files = sorted(Path(folder).glob(pattern), key=lambda p: (p.stat().st_mtime, p.name))
    if not files:
        raise FileNotFoundError(f"{folder} に {pattern} が見つかりません")
    per_file, kept, sources, conflicts = [], {}, {}, []
    # 古いファイルから順に処理し、同じ文献番号は後(=新しいファイル)の内容で上書きする
    for f in files:
        rows = parse_text(read_text_file(f))
        per_file.append({"ファイル": f.name, "読み取り件数": len(rows),
                         "警告あり": sum(bool(r.warnings) for r in rows)})
        for r in rows:
            if r.doc_no in kept and (kept[r.doc_no].status != r.status or kept[r.doc_no].title != r.title):
                conflicts.append({"文献番号": r.doc_no, "採用ファイル": f.name,
                                  "旧ステータス": " / ".join(kept[r.doc_no].status), "新ステータス": " / ".join(r.status)})
            kept[r.doc_no] = r
            sources.setdefault(r.doc_no, [])
            if f.name not in sources[r.doc_no]:
                sources[r.doc_no].append(f.name)
    recs = []
    for doc_no, r in kept.items():
        d = r.to_dict()
        d["source_files"] = " ; ".join(sources[doc_no])
        recs.append(d)
    df = pd.DataFrame(recs)
    same_app = df.groupby("app_no")["doc_no"].apply(list)
    df["same_app_docs"] = [" ; ".join(x for x in same_app[a] if x != d) for a, d in zip(df["app_no"], df["doc_no"])]
    df = df.sort_values("publication_date", ascending=False, kind="stable").reset_index(drop=True)
    df["no"] = range(1, len(df) + 1)
    out = df[list(LIST_COLUMNS) + ["source_files", "same_app_docs"]].rename(
        columns={**LIST_COLUMNS, "source_files": "出典ファイル", "same_app_docs": "同一出願番号の別文献"})
    total = sum(p["読み取り件数"] for p in per_file)
    report = {"files": pd.DataFrame(per_file), "total_rows": total, "unique": len(out),
              "duplicates_removed": total - len(out), "conflicts": pd.DataFrame(conflicts)}
    return out, report


### セル4: 手順1の実行: 一覧の作成
- **作業**: 設定したフォルダの全 txt を読み、重複を除いた文献番号の一覧を作って Excel に保存する
- **入力**: 設定の INPUT_DIR 内の txt
- **出力**: numbers(一覧)、report(報告)、results/patent_numbers.xlsx

In [ ]:
import pandas as pd

# フォルダ内の全 txt を読み、文献番号で重複を除いた一覧(numbers)と、読み込みの報告(report)を作る。
numbers, report = build_list(INPUT_DIR, INPUT_PATTERN)
# 一覧を Excel に保存する。
numbers.to_excel(NUMBERS_XLSX, index=False)
# 件数の内訳を表示する。
print(f"読み取り合計 {report['total_rows']} 件 -> 重複除去後 {report['unique']} 件（除去 {report['duplicates_removed']} 件）-> {NUMBERS_XLSX}")
# ファイルごとの読み取り件数を表示する。0件のファイルは、書式が違う可能性がある。
report["files"]

### セル5: 確認: ファイル間で内容が違った行
- **作業**: 同じ文献番号でファイル間にステータス等の違いがあった行を表示する。新しいファイルの内容を採用済み
- **入力**: report
- **出力**: 表(空なら該当なし)

In [ ]:
report['conflicts']

### セル6: 確認: 読み取れなかった行
- **作業**: 文献番号や日付の書式が想定と違い、読み取りに警告が出た行を表示する
- **入力**: numbers
- **出力**: 表(空なら問題なし)

In [ ]:
numbers[numbers['警告'] != ''][['No.', '文献番号', '警告']]

### セル7: 確認: 同じ出願番号で別の文献番号の行
- **作業**: 同じ出願番号で文献番号が違う行(公開公報と登録公報など)を表示する。除外はしていないので、重複評価を避けたい場合の参考にする
- **入力**: numbers
- **出力**: 表

In [ ]:
numbers[numbers['同一出願番号の別文献'] != ''][['文献番号', '出願番号', '同一出願番号の別文献']]

## 手順2: Google Patents から本文を取得する

### セル8: プログラム2: HTML の取得と本文の抽出
- **作業**: 文献番号から Google Patents の日本語ページの HTML を取得して保存し、名称・要約・請求項・明細書を取り出す。全文は切り詰めずに JSON でも保存する(原文と、全角を半角に揃えた版)
- **入力**: なし(関数を定義するだけ)
- **出力**: なし(次のセルで使う関数 run などができる)

In [ ]:
"""プログラム2: 文献番号の表から Google Patents(日本語ページ)の HTML を取得し、本文を抽出して保存する。

作業の流れ:
 1. fetch         : 1 件分の URL にアクセスして HTML を取得し、results/html/ に保存する(取得済みは再利用)
 2. extract       : HTML から、名称・要約・請求項・明細書(段落番号つき)を取り出す
 3. save_fulltext : 全文を切り詰めずに JSON で保存する(原文と、全角を半角に揃えた NFKC 版)
 4. run           : 上の 1〜3 を表の全行に対して繰り返し、結果の表(Excel 出力用)を返す

python -m patent_search.fetch_google IN.xlsx OUT.xlsx [--html-dir results/html] [--limit N] [--delay 3]

- 取得した生の HTML は --html-dir に保存する（再実行時は取得済みをスキップ）。
- 本文は <section itemprop=...> / <div class="claim"> 等の構造から抽出する。
  Google 側の画面構造は予告なく変わるため、抽出に失敗した行は「抽出状況」に記録する。
- 注意: Google Patents の自動取得は利用規約上の制限を受け得る。件数を絞り、間隔を空けること。
"""

from __future__ import annotations

import argparse
import json
import re
import unicodedata
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://patents.google.com/patent/{id}/ja"
UA = "Mozilla/5.0 (patent_search research; contact: owner of this repository)"
EXCEL_CELL_MAX = 32000


def _text(node) -> str:
    """ブロック内のインライン要素(<u>, <sub> 等)は改行せず連結し、<br> だけを改行にする。"""
    if node is None:
        return ""
    node = BeautifulSoup(str(node), "lxml")
    for br in node.find_all("br"):
        br.replace_with("\n")
    text = node.get_text("")
    lines = (re.sub(r"[ \t\u3000]+", " ", l).strip() for l in text.splitlines())
    return "\n".join(l for l in lines if l)


# 作業: <section> から見出し(<h2>)を除いた本体部分だけを返す。
def _content(section):
    """見出し(<h2>)を除いた本体。"""
    return section.find(attrs={"itemprop": "content"}) or section if section else None


# 作業: Google Patents のページ構造(itemprop 属性)から、名称・要約・請求項・明細書を取り出す。
def extract(html: str) -> dict:
    s = BeautifulSoup(html, "lxml")
    title = s.find(attrs={"itemprop": "title"}) or s.find("h1")
    abstract = s.find("section", attrs={"itemprop": "abstract"})
    desc = s.find("section", attrs={"itemprop": "description"})
    claims_sec = s.find("section", attrs={"itemprop": "claims"})
    # 請求項は <li class="claim"> の中の <div class="claim" num="N"> が実体（li と div の二重取りを避ける）
    claims = []
    if claims_sec:
        for c in claims_sec.find_all("div", class_="claim"):
            claims.append(f"【請求項{c.get('num', len(claims) + 1)}】" + _text(c))
    paras = []
    if desc:
        for p in desc.find_all(class_="description-paragraph"):
            paras.append(f"[{p.get('num', '')}] " + _text(p) if p.get("num") else _text(p))
    out = {
        "title": re.sub(r"\s+", " ", _text(title)).replace(" - Google Patents", ""),
        "abstract": _text(_content(abstract)),
        "claims": "\n".join(claims),
        "description": "\n".join(paras) or _text(_content(desc)),
        "claim_count": len(claims),
    }
    missing = [k for k in ("title", "abstract", "claims", "description") if not out[k]]
    out["extract_status"] = "OK" if not missing else "欠落: " + ",".join(missing)
    return out


# 作業: 1 件の HTML を取得する。保存済みならそれを使い、なければ通信して保存する。
def fetch(session: requests.Session, pid: str, html_dir: Path, delay: float):
    """(html, 使った ID, HTTP 状態) を返す。ID は 末尾の種別コードを外した候補も試す。"""
    for cand in dict.fromkeys([pid, re.sub(r"[A-Z]\d?$", "", pid)]):
        path = html_dir / f"{cand}.html"
        if path.exists():
            return path.read_text(encoding="utf-8"), cand, "cache"
        time.sleep(delay)
        r = session.get(URL.format(id=cand), timeout=60, headers={"Accept-Language": "ja"})
        if r.status_code == 200:
            r.encoding = "utf-8"
            path.write_text(r.text, encoding="utf-8")
            return r.text, cand, "200"
        status = str(r.status_code)
    return "", pid, status


def nfkc(text: str) -> str:
    """全角英数字・記号を半角に揃える(NFKC)。検索や LLM 入力での表記ゆれ対策。"""
    return unicodedata.normalize("NFKC", text)


def save_fulltext(text_dir: Path, pid: str, x: dict) -> Path:
    """切り詰めなしの全文(原文と NFKC 正規化)を JSON で保存する。"""
    path = text_dir / f"{pid}.json"
    data = {k: x[k] for k in ("title", "abstract", "claims", "description")}
    data.update({k + "_nfkc": nfkc(x[k]) for k in ("title", "abstract", "claims", "description")})
    path.write_text(json.dumps(data, ensure_ascii=False, indent=1), encoding="utf-8")
    return path


def run(df: pd.DataFrame, html_dir: Path, delay: float = 3.0, text_dir: Path | None = None) -> pd.DataFrame:
    """文献番号の表(Google Patents ID(推定) 列を持つ)から取得・抽出し、結果の表を返す。

    text_dir を渡すと、全文を JSON で保存する(Excel のセル上限で切り詰められるため、LLM 評価にはこちらを使う)。
    """
    html_dir = Path(html_dir)
    html_dir.mkdir(parents=True, exist_ok=True)
    text_dir = Path(text_dir) if text_dir else html_dir.parent / "text"
    text_dir.mkdir(parents=True, exist_ok=True)
    sess = requests.Session()
    sess.headers["User-Agent"] = UA
    rows = []
    for _, r in df.iterrows():
        pid = r.get("Google Patents ID(推定)")
        rec = {"文献番号": r["文献番号"], "ID": pid}
        if not isinstance(pid, str) or not pid:
            rec.update({"取得状況": "IDなし"})
        else:
            try:
                html, used, status = fetch(sess, pid, html_dir, delay)
            except requests.RequestException as e:
                html, used, status = "", pid, f"error: {type(e).__name__}"
            rec.update({"使用ID": used, "取得状況": status})
            if html:
                x = extract(html)
                rec.update({"名称(Google)": x["title"], "要約": x["abstract"],
                            "請求項数": x["claim_count"], "請求項": x["claims"],
                            "明細書": x["description"], "明細書文字数": len(x["description"]),
                            "名称(正規化)": nfkc(x["title"]), "全文ファイル": str(save_fulltext(text_dir, used, x)),
                            "抽出状況": x["extract_status"]})
                # Excel のセルは約 3 万 2 千文字までなので、超える分は切り詰める(全文は JSON と HTML に残る)
                if len(x["description"]) > EXCEL_CELL_MAX:
                    rec["明細書"] = x["description"][:EXCEL_CELL_MAX]
                    rec["抽出状況"] += " / 明細書を切り詰め(全文は全文ファイル)"
                if len(x["claims"]) > EXCEL_CELL_MAX:
                    rec["請求項"] = x["claims"][:EXCEL_CELL_MAX]
                    rec["抽出状況"] += " / 請求項を切り詰め"
        rows.append(rec)
        print(rec["文献番号"], rec.get("取得状況"), rec.get("抽出状況", ""), flush=True)
    return pd.DataFrame(rows)


### セル9: 手順2の実行: 取得と抽出
- **作業**: 一覧の先頭から LIMIT 件について、HTML を取得し本文を抽出して Excel に保存する。取得済みの HTML は再利用する
- **入力**: numbers、設定の LIMIT・DELAY
- **出力**: result(表)、results/patents_text.xlsx、results/html/、results/text/

In [ ]:
# LIMIT が None なら全件、数字ならその件数だけを対象にする。
targets = numbers if LIMIT is None else numbers.head(LIMIT)
# 取得・抽出を実行し、結果の表を受け取る。
result = run(targets, Path(HTML_DIR), DELAY, Path(TEXT_DIR))
# 結果を Excel に保存する。
result.to_excel(RESULT_XLSX, index=False)
print('->', RESULT_XLSX)
# 取得状況・抽出状況などの要点だけを表示する。
result[[c for c in ['文献番号', 'ID', '使用ID', '取得状況', '抽出状況', '請求項数'] if c in result]]

### セル10: 確認: 取得・抽出に失敗した行
- **作業**: 取得状況が 200 / cache 以外、または抽出状況が OK 以外の行を表示する
- **入力**: result
- **出力**: 表(空なら全件成功)

In [ ]:
result[(result['取得状況'].isin(['200', 'cache']) == False) | (result.get('抽出状況', 'OK') != 'OK')]

## 手順3: LLM による関連性の評価

### セル11: プログラム3: 関連性の評価
- **作業**: 調べたいことの関連語の周辺を本文から抜粋し、判定基準と一緒に LLM へのプロンプトを作って判定させる。判定は JSON の形式に制約され、「根拠の引用」が原文に実在するかも自動で確認する
- **入力**: なし(関数を定義するだけ)
- **出力**: なし(次の 2 つのセルで使う関数ができる)

In [ ]:
"""プログラム3: 特許本文と「調べたいこと」の関連性を、ローカル LLM で評価する。

作業の流れ:
 1. keyword_snippets: 本文から、調べたいことの関連語の周辺を抜粋する(本文が長いため)
 2. build_prompt    : 調べたいこと + 判定基準 + 特許の本文(抜粋)から、LLM に渡す文章を作る
 3. バックエンド    : ollama_backend(この PC) / guidance_backend(サーバー)で LLM に判定させる
 4. check_evidence  : LLM が出した「根拠の引用」が原文に実在するかを確認する
 5. evaluate_table  : 上を全件に繰り返し、関連度の高い順の結果表を返す(発明内容概要・出願企業も付ける)
 6. save_evaluation_excel: 結果表を Excel に保存する(文献番号は Google Patents へのリンク)

- 判定結果は JSON スキーマで制約する（Ollama: format 指定 / Hugging Face: guidance の json 文法）。
  どちらのバックエンドでも同じプロンプト・同じスキーマなので、サーバー移植時はバックエンドだけ差し替える。
- LLM が出した「根拠の引用」が本当に原文にあるかを機械的に確認し、結果に付ける（幻覚の検出）。
"""

from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass, field
from typing import Callable

SCHEMA = {
    "type": "object",
    "properties": {
        "score": {"type": "integer", "enum": [0, 1, 2, 3]},
        "judgement": {"type": "string", "enum": ["直接関連", "関連あり", "わずかに関連", "無関係"]},
        "reason": {"type": "string", "maxLength": 300},
        "summary": {"type": "string", "maxLength": 220},
        "evidence": {"type": "array", "items": {"type": "string", "maxLength": 200}, "maxItems": 3},
    },
    "required": ["score", "judgement", "reason", "summary", "evidence"],
    "additionalProperties": False,
}

SYSTEM = "あなたは特許調査の専門家です。与えられた特許の本文が、指定された調査テーマに関連するかを、根拠を示して厳密に判定します。"

USER_TEMPLATE = """# 調査テーマ
{name}

## テーマの定義
{definition}

## 関連語（同義語・日英表記）
{keywords}

## 含める条件（関連とみなす）
{include}

## 除外する条件（関連とみなさない）
{exclude}

# 判定基準
- 3 (直接関連): 特許の中心的な内容（請求項・要約）が、このテーマの手法・対象そのものである
- 2 (関連あり): テーマの手法・対象を、明細書や実施例で実際に使用・言及している（請求項の中心ではない）
- 1 (わずかに関連): 関連語が出るが、一般的な背景説明や列挙の一部にとどまる
- 0 (無関係): 関連しない

# 対象特許
文献番号: {doc_no}
名称: {title}

## 要約
{abstract}

## 請求項
{claims}

## 明細書（関連語の周辺の抜粋 / 先頭部分）
{description}

# 出力
JSON のみを出力する。summary には、この特許の発明の内容（何を、どうする発明か）を、専門外の人にも分かる日本語で100〜150字にまとめる（要約と請求項に基づき、本文にないことは書かない）。
evidence には、上の本文からそのまま抜き出した短い引用（最大3件、各200字以内、改変禁止）を入れる。
根拠が本文にない場合は evidence を空配列にし、score は 0 か 1 にする。"""


@dataclass
class Topic:
    name: str
    definition: str
    keywords: list[str] = field(default_factory=list)
    include: str = ""
    exclude: str = ""


def nfkc(s: str) -> str:
    return unicodedata.normalize("NFKC", s)


def _keyword_patterns(keywords: list[str]) -> list[re.Pattern]:
    """通常の語は大文字小文字を区別しない。SHAPE・NAI・DMS のような大文字だけの短い略語は、
    shape(形状)・naive・DGSHAPE 等への誤一致を避けるため、大文字小文字を区別し前後に英数字がある場合は除外する。"""
    ci, strict = [], []
    for k in map(nfkc, keywords):
        if re.fullmatch(r"[A-Z0-9]{2,6}", k):
            strict.append(r"(?<![A-Za-z0-9])" + re.escape(k) + r"(?![A-Za-z0-9])")
        else:
            ci.append(re.escape(k))
    pats = []
    if ci:
        pats.append(re.compile("|".join(ci), re.IGNORECASE))
    if strict:
        pats.append(re.compile("|".join(strict)))
    return pats


# 作業: 関連語の前後を抜粋する。近い箇所は 1 つにまとめ、全ヒット数も返す。
def keyword_snippets(text: str, keywords: list[str], width: int = 150, max_snippets: int = 8) -> tuple[list[str], int]:
    """関連語の前後 width 文字を抜粋する(重なる範囲は結合)。(抜粋, 全ヒット数) を返す。"""
    if not keywords:
        return [], 0
    hits = sorted(m.span() for pat in _keyword_patterns(keywords) for m in pat.finditer(text))
    spans: list[list[int]] = []
    for a, b in hits:
        a, b = max(0, a - width), min(len(text), b + width)
        if spans and a <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], b)
        else:
            spans.append([a, b])
    return [text[a:b].replace("\n", " ") for a, b in spans[:max_snippets]], len(hits)


# 作業: 調べたいこと・判定基準・特許本文を、1 つのプロンプトに組み立てる。
def build_prompt(topic: Topic, doc_no: str, rec: dict, max_chars: int = 6000) -> tuple[str, int]:
    """rec は全文JSON(*_nfkc キーを使う)。max_chars で本文の長さを抑える。(プロンプト, 関連語ヒット数)"""
    desc = rec["description_nfkc"]
    snippets, hits = keyword_snippets(desc, topic.keywords)
    body = "\n---\n".join(snippets) if snippets else desc[:1500]
    claims = rec["claims_nfkc"]
    prompt = USER_TEMPLATE.format(
        name=topic.name, definition=topic.definition.strip(),
        keywords="、".join(topic.keywords) or "(なし)", include=topic.include.strip() or "(指定なし)",
        exclude=topic.exclude.strip() or "(指定なし)", doc_no=doc_no, title=rec["title_nfkc"],
        abstract=rec["abstract_nfkc"][:1500], claims=claims[: max_chars // 2], description=body[: max_chars // 2],
    )
    return prompt, hits


# ---- バックエンド: (system, user, schema) -> JSON 文字列 -------------------------------

# 作業: Ollama(この PC)に問い合わせる関数を作る。出力は JSON スキーマで制約される。
def ollama_backend(model: str = "qwen3:14b", host: str = "http://localhost:11434", num_ctx: int = 8192) -> Callable:
    import requests

    def run(system: str, user: str, schema: dict) -> str:
        r = requests.post(f"{host}/api/chat", timeout=600, json={
            "model": model, "stream": False, "think": False, "format": schema,
            "options": {"temperature": 0, "num_ctx": num_ctx},
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        })
        r.raise_for_status()
        return r.json()["message"]["content"]

    return run


# 作業: guidance(Hugging Face のモデル、サーバー用)に問い合わせる関数を作る。出力は JSON 文法で制約される。
def guidance_backend(lm) -> Callable:
    """lm: guidance のモデル。例: guidance.models.Transformers("Qwen/Qwen3-14B", device_map="auto")。"""
    from guidance import assistant, system as system_role, user as user_role
    from guidance import json as gjson

    def run(system: str, user: str, schema: dict) -> str:
        m = lm
        with system_role():
            m += system
        with user_role():
            m += user
        with assistant():
            m += gjson(name="out", schema=schema)
        return m["out"]

    return run


# ---- 評価 ----------------------------------------------------------------------------

# 作業: LLM の引用が、原文(全角半角・空白の違いを無視)に含まれるかを確認する。
def check_evidence(evidence: list[str], source: str) -> str:
    """引用が原文(NFKC・空白除去)に含まれるかを確認する。"""
    norm = lambda s: re.sub(r"\s+", "", nfkc(s))
    if not evidence:
        return "引用なし"
    src = norm(source)
    ok = sum(norm(e) in src for e in evidence)
    return "全て原文に存在" if ok == len(evidence) else f"原文に無い引用あり({len(evidence) - ok}/{len(evidence)})"


# 作業: 1 件を評価する(プロンプト作成 → LLM → JSON の読み取り → 引用の検証)。
def evaluate_one(backend: Callable, topic: Topic, doc_no: str, rec: dict, max_chars: int = 6000) -> dict:
    prompt, hits = build_prompt(topic, doc_no, rec, max_chars)
    try:
        out = json.loads(backend(SYSTEM, prompt, SCHEMA))
    except (json.JSONDecodeError, KeyError, ValueError) as e:
        return {"score": None, "judgement": "判定失敗", "reason": f"{type(e).__name__}: {e}", "summary": "",
                "evidence": [], "evidence_check": "", "keyword_hits": hits}
    source = rec["abstract_nfkc"] + rec["claims_nfkc"] + rec["description_nfkc"]
    out["evidence_check"] = check_evidence(out.get("evidence", []), source)
    out["keyword_hits"] = hits
    return out


# 最終の Excel に出す列と順番。
FINAL_COLUMNS = ["文献番号", "判定", "関連度", "関連語ヒット数", "理由", "引用の検証", "根拠の引用",
                 "発明の名称", "出願人・権利者", "出願企業", "ステータス", "発明内容概要"]

# 法人格・会社の種類を表す語。名前の末尾から取り除く(途中の「インク」等は残すため、末尾だけ)。
_CORP_SUFFIXES = [
    "インコーポレイテッド", "インコーポレイティド", "インコーポレーテッド", "インコーポレイテツド", "コーポレーション", "コーポレイション",
    "ライアビリティ", "リミテッド", "リミティド", "カンパニー", "エルエルシー", "ゲーエムベーハー", "アクチェンゲゼルシャフト",
    "ソシエテアノニム", "アーゲー", "インク",
]
# 法人格を表す漢字の語。名前のどこにあっても取り除く(前株・後株の両方に対応)。
_CORP_KANJI = ["株式会社", "有限会社", "合同会社", "股份有限公司", "有限公司"]


def company_name(applicant, aliases: dict | None = None) -> str:
    """出願人の欄から、出願企業名を作る。

    全角半角をそろえ、区切り(空白・中黒・読点)と法人格の語(株式会社、インコーポレイテッド など)を取り除く。
    aliases({正規化した名前: 表示名})に一致すれば、その表示名を使う。
    出願人が複数の場合、J-PlatPat の一覧に載っているのは先頭の 1 者だけ(「他あり」)。
    """
    if not isinstance(applicant, str) or not applicant.strip():
        return ""
    # 特許庁の外字表記「▲ふん▼」は「份」(股份有限公司)のこと
    name = nfkc(applicant).replace("▲ふん▼", "份")
    for w in _CORP_KANJI:
        name = name.replace(w, "")
    name = re.sub(r"[\s・･,，.．、]+", "", name)
    # 末尾の法人格の語を、なくなるまで繰り返し取り除く(名前全体が消える場合は止める)
    changed = True
    while changed:
        changed = False
        for w in _CORP_SUFFIXES:
            if name.endswith(w) and len(name) > len(w):
                name = name[: -len(w)]
                changed = True
    return (aliases or {}).get(name, name)


# 作業: 文献番号表の各行を、全文 JSON(text_dir/{ID}.json)で評価し、FINAL_COLUMNS の順の結果表を返す。
def evaluate_table(backend: Callable, topic: Topic, numbers, text_dir, max_chars: int = 6000, aliases: dict | None = None):
    from pathlib import Path

    import pandas as pd

    rows = []
    for _, r in numbers.iterrows():
        pid = r.get("Google Patents ID(推定)")
        path = Path(text_dir) / f"{pid}.json"
        applicant = r.get("出願人/権利者")
        more = "（他あり）" if r.get("出願人(他あり)") is True or str(r.get("出願人(他あり)")) == "True" else ""
        base = {"文献番号": r["文献番号"], "リンク": r.get("Google Patents URL(推定)"), "発明の名称": r.get("発明の名称"),
                "出願人・権利者": f"{applicant}{more}" if isinstance(applicant, str) else "",
                "出願企業": company_name(applicant, aliases), "ステータス": r.get("ステータス")}
        if not path.exists():
            rows.append({**base, "判定": "本文なし(未取得)"})
            continue
        rec = json.loads(path.read_text(encoding="utf-8"))
        try:
            o = evaluate_one(backend, topic, r["文献番号"], rec, max_chars)
        # 接続断などでも、1 件の失敗で全体を止めず、失敗として記録して次へ進む
        except Exception as e:
            o = {"score": None, "judgement": "判定失敗", "reason": f"{type(e).__name__}: {e}", "summary": "",
                 "evidence": [], "evidence_check": "", "keyword_hits": None}
        # LLM の概要が空なら、Google Patents の要約(公式)の先頭で代用し、その旨を付ける
        summary = o.get("summary") or (("（要約より）" + rec["abstract_nfkc"][:150]) if rec.get("abstract_nfkc") else "")
        rows.append({**base, "関連度": o["score"], "判定": o["judgement"], "理由": o["reason"],
                     "根拠の引用": " / ".join(o["evidence"]), "引用の検証": o["evidence_check"],
                     "関連語ヒット数": o["keyword_hits"], "発明内容概要": summary})
        print(r["文献番号"], o["score"], o["judgement"], flush=True)
    df = pd.DataFrame(rows)
    for c in FINAL_COLUMNS + ["リンク"]:
        if c not in df:
            df[c] = None
    df = df.sort_values("関連度", ascending=False, na_position="last", kind="stable")
    return df[FINAL_COLUMNS + ["リンク"]].reset_index(drop=True)


# 作業: 結果表を Excel に保存する。文献番号をクリックで Google Patents が開くリンクにし、見やすく整える。
def save_evaluation_excel(df, path) -> None:
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    out = df[FINAL_COLUMNS]
    with __import__("pandas").ExcelWriter(path, engine="openpyxl") as w:
        out.to_excel(w, index=False, sheet_name="評価結果")
        ws = w.sheets["評価結果"]
        widths = {"文献番号": 18, "判定": 14, "関連度": 8, "関連語ヒット数": 10, "理由": 50, "引用の検証": 20,
                  "根拠の引用": 60, "発明の名称": 40, "出願人・権利者": 28, "出願企業": 20, "ステータス": 22, "発明内容概要": 60}
        wrap = {"理由", "根拠の引用", "発明の名称", "出願人・権利者", "発明内容概要"}
        for j, c in enumerate(FINAL_COLUMNS, start=1):
            ws.column_dimensions[get_column_letter(j)].width = widths[c]
            head = ws.cell(row=1, column=j)
            head.font = Font(bold=True)
            head.fill = PatternFill("solid", fgColor="DDDDDD")
            for i in range(2, len(out) + 2):
                ws.cell(row=i, column=j).alignment = Alignment(wrap_text=c in wrap, vertical="top")
        # 文献番号のセルに、Google Patents へのリンクを付ける
        for i, url in enumerate(df["リンク"], start=2):
            if isinstance(url, str) and url:
                cell = ws.cell(row=i, column=1)
                cell.hyperlink = url
                cell.font = Font(color="0563C1", underline="single")
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions


### セル12: LLM の準備とプロンプトの確認
- **作業**: 設定の BACKEND に応じて LLM に問い合わせる準備をし、調べたいことをまとめ、実際に LLM に渡る文章(プロンプト)の見本を表示する。ollama は Ollama の起動と `ollama pull qwen3:14b` 済みであること。guidance は `pip install guidance transformers torch accelerate` が必要で、モデルは初回に Hugging Face から取得される
- **入力**: 設定の BACKEND・TOPIC_*、results/text/ の全文 JSON
- **出力**: backend、topic、プロンプトの見本(画面表示)

In [ ]:
# 設定の BACKEND に応じて、LLM に問い合わせる関数(backend)を用意する。
if BACKEND == "ollama":
    backend = ollama_backend(OLLAMA_MODEL, num_ctx=OLLAMA_NUM_CTX)
elif BACKEND == "guidance":
    from guidance import models
    backend = guidance_backend(models.Transformers(HF_MODEL_ID, device_map="auto"))
else:
    raise ValueError(BACKEND)
# 設定の「調べたいこと」を 1 つにまとめる。
topic = Topic(TOPIC_NAME, TOPIC_DEFINITION, TOPIC_KEYWORDS, TOPIC_INCLUDE, TOPIC_EXCLUDE)
# 手順2で保存した全文 JSON のうち、1 件目をプロンプトの見本に使う。
sample = next(Path(TEXT_DIR).glob("*.json"))
prompt, _ = build_prompt(topic, "（例）" + sample.stem, json.loads(sample.read_text(encoding="utf-8")), MAX_CHARS)
# LLM に実際に渡る文章を表示する(内容を目で確認する)。
print(prompt)

### セル13: 手順3の実行: 評価
- **作業**: 一覧の先頭から EVAL_LIMIT 件を LLM で評価し、関連度の高い順の Excel にする。列は、文献番号(Google Patents へのリンク)、判定、関連度、関連語ヒット数、理由、引用の検証、根拠の引用、発明の名称、出願人・権利者、出願企業、ステータス、発明内容概要
- **入力**: numbers、results/text/ の全文 JSON、backend、topic
- **出力**: evaluation(表)、results/evaluation.xlsx

In [ ]:
# EVAL_LIMIT が None なら全件、数字ならその件数だけを対象にする。
targets3 = numbers if EVAL_LIMIT is None else numbers.head(EVAL_LIMIT)
# LLM で 1 件ずつ評価し、結果の表を受け取る。
evaluation = evaluate_table(backend, topic, targets3, TEXT_DIR, MAX_CHARS, COMPANY_ALIASES)
# 結果を Excel に保存する(文献番号をクリックで Google Patents が開く。列と順番は固定)。
save_evaluation_excel(evaluation, RESULT_EVAL_XLSX)
print('->', RESULT_EVAL_XLSX)
# 結果の表を画面に表示する(リンク列は Excel 側でだけ使うので除く)。
evaluation.drop(columns=["リンク"])